#### Transform Orders Data - Explode Arrays
##### 1. Access elements from the JSO Objects
##### 2. Deduplicate Array Elements
##### 3. Explode Arrays
##### 4. Write the Transformed Data to Silver Schema

In [0]:
df = spark.read.table('gizmobox_catalog_noori.silver.orders')
display(df)

In [0]:
df.select(df.json_value.customer_id.alias('customer_id'), 
          df.json_value.order_status.alias('order_status'), 
          df.json_value.order_id.alias('order_id'), 
          df.json_value.transaction_timestamp.alias('transaction_timestamp'), 
          df.json_value.payment_method.alias('payment_method'), 
          df.json_value.total_amount.alias('total_amount'),
          df.json_value.items.alias('order_items')
          ).display()

In [0]:
from pyspark.sql.functions import array_distinct
df.select(df.json_value.customer_id.alias('customer_id'), 
          df.json_value.order_status.alias('order_status'), 
          df.json_value.order_id.alias('order_id'), 
          df.json_value.transaction_timestamp.alias('transaction_timestamp'), 
          df.json_value.payment_method.alias('payment_method'), 
          df.json_value.total_amount.alias('total_amount'),
          array_distinct(df.json_value.items).alias('order_items')
          ).display()

In [0]:
from pyspark.sql.functions import array_distinct, explode
df_exploded = df.select(df.json_value.customer_id.alias('customer_id'), 
          df.json_value.order_status.alias('order_status'), 
          df.json_value.order_id.alias('order_id'), 
          df.json_value.transaction_timestamp.alias('transaction_timestamp'), 
          df.json_value.payment_method.alias('payment_method'), 
          df.json_value.total_amount.alias('total_amount'),
          explode(array_distinct(df.json_value.items)).alias('item')
          )
display(df_exploded)

In [0]:
from pyspark.sql.functions import  col 
df_formatted = df_exploded.select(df_exploded.customer_id, 
          df_exploded.order_status, 
          df_exploded.order_id, 
          df_exploded.transaction_timestamp, 
          df_exploded.payment_method, 
          df_exploded.total_amount,
          df_exploded.item.item_id.alias('item_id'),
          col('item.name').alias('item_name'),          
          df_exploded.item.price.alias('item_price'),
          df_exploded.item.quantity.alias('item_quantity'),
          df_exploded.item.details.brand.alias('brand'),
          df_exploded.item.details.color.alias('color')
          )
display(df_formatted)

In [0]:
df_formatted.writeTo('gizmobox_catalog_noori.silver.orders_details').createOrReplace()

In [0]:
spark.read.table('gizmobox_catalog_noori.silver.orders_details').display()